# 🌍 NLLB-200 (1.3B) Hinglish-to-English Seq2Seq LoRA Fine-Tuning
### **Meta AI's No Language Left Behind (NLLB-200-distilled-1.3B)**

This notebook fine-tunes **`facebook/nllb-200-distilled-1.3B`** for **Romanized Hinglish-to-English Neural Machine Translation** on Google Colab (Free Tesla T4 GPU).

#### **Why NLLB-200 is Ideal for this Task:**
- **Purpose-Built Seq2Seq Architecture**: Cross-attention natively conditions generation on the full source sentence.
- **200+ Language Pre-Training**: Massive multilingual and Indic representation, easily adapting to code-mixed Latin transliteration.
- **Compact 1.3B Parameter Footprint**: Runs in under **2.5 GB VRAM** in 4-bit QLoRA and trains in only **~20-30 minutes** on Colab T4!
- **Directly Deployable**: Runs locally on an RTX 3050 Laptop GPU (6GB VRAM) without out-of-memory errors.

## Step 1: Verify GPU Environment

In [ ]:
!nvidia-smi

## Step 2: Clone or Pull Project Repository

In [ ]:
import os
if not os.path.exists('/content/NLP'):
    !git clone https://github.com/Nickhasntlost/NLP.git /content/NLP
    %cd /content/NLP
else:
    %cd /content/NLP
    !git pull origin main

## Step 3: Install Required Libraries

In [ ]:
# Uninstall torchao if present to avoid PyTorch conflicts
!pip uninstall -y torchao -q
!pip install -q transformers peft accelerate bitsandbytes sacrebleu evaluate rouge_score bert_score datasets

## Step 4: Fine-Tune NLLB-200 (1.3B) with Seq2Seq LoRA
Trains on the 9,769 web-scraped social media corpus pairs for 3 epochs with beam search evaluation.

In [ ]:
!python finetune/finetune_nllb.py \
    --base_model facebook/nllb-200-distilled-1.3B \
    --epochs 3 \
    --batch_size 8 \
    --grad_accum 2 \
    --lr 2e-4 \
    --lora_r 32 \
    --max_length 128

## Step 5: Multi-Model Benchmark Evaluation
Evaluates NLLB-200 alongside mT5-Small, Sarvam-1, and published literature baselines across BLEU, chrF++, ROUGE, and BERTScore.

In [ ]:
!python finetune/evaluate_colab.py --model nllb --limit 150

## Step 6: Test Interactive Translation

In [ ]:
from finetune.translate_nllb import NLLBTranslator

translator = NLLBTranslator()

test_sentences = [
    "kya chal raha hai bhai",
    "yeh video sach me bahut accha tha",
    "mujhe kal subah jaldi uthna hai",
    "aaj mausam kitna badhiya hai",
    "sir aap bahut achha padhate ho",
]

for s in test_sentences:
    res = translator.translate(s)
    print(f"{res['source']:<35} -> {res['translation']} ({res['latency_ms']} ms)")

## Step 7: Package & Download Adapter ZIP
Packages the trained adapter into `nllb_hinglish_lora.zip` and automatically downloads it to your computer.

In [ ]:
import os
!zip -r nllb_hinglish_lora.zip finetune/outputs/nllb_hinglish_lora/ -x "*.pt" "*optimizer*" "*scheduler*"

from google.colab import files
if os.path.exists('nllb_hinglish_lora.zip'):
    files.download('nllb_hinglish_lora.zip')
    print('✅ Adapter download initiated!')